In [1]:
import pandas as pd

file_path = "kaggle_recipes.csv"
recipe_df = pd.read_csv(file_path)

recipe_df.head()


,Unnamed: 0,recipe_id,recipe_name,aver_rate,image_url,review_nums,calories,fat,carbohydrates,protein,cholesterol,sodium,fiber,ingredients_list
0,0,222388,Homemade Bacon,5.00,https://images.media-allrecipes.com/userphotos...,3,15,36,1,42,21,81,2,"['pork belly', 'smoked paprika', 'kosher salt'..."
1,1,240488,"Pork Loin, Apples, and Sauerkraut",4.76,https://images.media-allrecipes.com/userphotos...,29,19,18,10,73,33,104,41,"['sauerkraut drained', 'Granny Smith apples sl..."
2,2,218939,Foolproof Rosemary Chicken Wings,4.57,https://images.media-allrecipes.com/userphotos...,12,17,36,2,48,24,31,4,"['chicken wings', 'sprigs rosemary', 'head gar..."
3,3,87211,Chicken Pesto Paninis,4.62,https://images.media-allrecipes.com/userphotos...,163,32,45,20,65,20,43,18,"['focaccia bread quartered', 'prepared basil p..."
4,4,245714,Potato Bacon Pizza,4.50,https://images.media-allrecipes.com/userphotos...,2,8,12,5,14,7,8,3,"['red potatoes', 'strips bacon', 'Sauce:', 'he..."


In [2]:
recipe_df.isnull().sum()

Unnamed: 0          0
recipe_id           0
recipe_name         0
aver_rate           0
image_url           0
review_nums         0
calories            0
fat                 0
carbohydrates       0
protein             0
cholesterol         0
sodium              0
fiber               0
ingredients_list    0
dtype: int64

In [3]:
recipe_df['ingredients_list'][0]

"['pork belly', 'smoked paprika', 'kosher salt', 'ground black pepper']"

In [4]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

In [5]:
# Preprocess Ingredients
vectorizer = TfidfVectorizer()
X_ingredients = vectorizer.fit_transform(recipe_df['ingredients_list'])

In [6]:
# Normalize Numerical Features
scaler = StandardScaler()
X_numerical = scaler.fit_transform(recipe_df[['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber']])

In [7]:
print(X_numerical)

[[-0.1317045   0.46001924 -1.15482863 ...  0.04256474  1.13990476
  -0.76393724]
 [ 0.23857551 -0.33625589 -0.01920347 ...  0.52871248  1.59202345
   2.53220175]
 [ 0.05343551  0.46001924 -1.02864806 ...  0.16410167  0.15703804
  -0.59490447]
 ...
 [-0.77969453 -0.82286847 -0.27156462 ... -0.68665688 -0.39336732
  -0.25683894]
 [ 0.33114552  0.32730672  0.73787996 ...  0.20461398 -0.31473798
  -0.51038809]
 [-1.33511455 -0.9998185  -1.02864806 ... -0.80819381 -0.13782197
  -0.42587171]]


In [8]:
# Combine Features
X_combined = np.hstack([X_numerical, X_ingredients.toarray()])
print(X_combined)
# Train KNN Model
knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(X_combined)

[[-0.1317045   0.46001924 -1.15482863 ...  0.          0.
   0.        ]
 [ 0.23857551 -0.33625589 -0.01920347 ...  0.          0.
   0.        ]
 [ 0.05343551  0.46001924 -1.02864806 ...  0.          0.
   0.        ]
 ...
 [-0.77969453 -0.82286847 -0.27156462 ...  0.          0.
   0.        ]
 [ 0.33114552  0.32730672  0.73787996 ...  0.          0.
   0.        ]
 [-1.33511455 -0.9998185  -1.02864806 ...  0.          0.
   0.        ]]


NearestNeighbors(metric='euclidean', n_neighbors=3)

In [9]:
# Function to Recommend Recipes
def recommend_recipes(input_features):
    input_features_scaled = scaler.transform([input_features[:7]])
    input_ingredients_transformed = vectorizer.transform([input_features[7]])
    input_combined = np.hstack([input_features_scaled, input_ingredients_transformed.toarray()])
    distances, indices = knn.kneighbors(input_combined)
    recommendations = recipe_df.iloc[indices[0]]
    return recommendations[['recipe_name', 'ingredients_list', 'image_url']]

# Example Input
input_features = [15, 36, 1, 42, 21, 81, 2, 'pork belly, smoked paprika, kosher salt']
recommendations = recommend_recipes(input_features)
recommendations

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,recipe_name,ingredients_list,image_url
0,Homemade Bacon,"['pork belly', 'smoked paprika', 'kosher salt'...",https://images.media-allrecipes.com/userphotos...
3507,Char Siu (Chinese Barbeque Pork),"['honey', 'Shaoxing cooking wine', 'hoisin sau...",https://images.media-allrecipes.com/userphotos...
1926,Adobo Twist,"['vegetable oil', 'onion', 'head garlic', 'por...",https://images.media-allrecipes.com/userphotos...


In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

# Load the dataset
file_path = "kaggle_recipes.csv"
recipe_df = pd.read_csv(file_path)

# Preprocess Ingredients (vectorize)
vectorizer = TfidfVectorizer()
X_ingredients = vectorizer.fit_transform(recipe_df['ingredients_list'])

# Normalize Numerical Features
scaler = StandardScaler()
X_numerical = scaler.fit_transform(recipe_df[['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber']])

# Combine Numerical and Ingredient Features
X_combined = np.hstack([X_numerical, X_ingredients.toarray()])

# Use recipe names as labels for classification
y = recipe_df['recipe_name']

# Split the data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Initialize and fit the KNN model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predict on the test set
y_pred = knn.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Precision (average='weighted' handles multiclass classification)
precision = precision_score(y_test, y_pred, average='weighted')
print(f"Precision: {precision:.2f}")

# Recall
recall = recall_score(y_test, y_pred, average='weighted')
print(f"Recall: {recall:.2f}")

# F1-Score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"F1-Score: {f1:.2f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))


# # Function to recommend recipes based on input features
# def recommend_recipes(input_features):
#     # Separate numeric and ingredient features
#     input_numerical = np.array(input_features[:7]).reshape(1, -1)  # First 7 are numeric features
#     input_ingredients = [input_features[7]]  # Last one is the ingredient list (in string form)
    
#     # Convert numeric features to DataFrame (with original column names)
#     input_numerical_df = pd.DataFrame(input_numerical, columns=['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber'])
    
#     # Scale the numeric features and transform the ingredient list
#     input_features_scaled = scaler.transform(input_numerical_df)
#     input_ingredients_transformed = vectorizer.transform(input_ingredients)  # Vectorize the ingredients list
    
#     # Combine the numeric and ingredient features
#     input_combined = np.hstack([input_features_scaled, input_ingredients_transformed.toarray()])
    
#     # Get recommendations (top 3 nearest neighbors)
#     distances, indices = knn.kneighbors(input_combined)
    
#     # Return recommended recipe details
#     recommendations = recipe_df.iloc[indices[0]]
#     return recommendations[['recipe_name', 'ingredients_list', 'image_url']]

# # Example Input (numeric features + ingredient list)
# input_features = [15, 36, 1, 42, 21, 81, 2, 'pork belly, smoked paprika, kosher salt']
# recommendations = recommend_recipes(input_features)


# # Print recommended recipes
# print(recommendations)


Accuracy: 0.00


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Precision: 0.00


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Recall: 0.00
F1-Score: 0.00
Confusion Matrix:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
Classification Report:


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{

                                                                                                                     precision    recall  f1-score   support

                                                                                "515" Southern-Style St. Louis Ribs       0.00      0.00      0.00         0
                                                                                      "Black Friday" Turkey Pot Pie       0.00      0.00      0.00         1
                                                                                        "CranCrack" Cranberry Salad       0.00      0.00      0.00         0
                                                                                                 "Doritos®" Popcorn       0.00      0.00      0.00         1
                                                             "Fruit Smoothie" No-Bake Cheesecake from PHILADELPHIA®       0.00      0.00      0.00         0
                                                         

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predict on the test set
y_pred = knn.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Precision, Recall, and F1-Score with zero_division=1
precision = precision_score(y_test, y_pred, average='weighted', zero_division=1)
print(f"Precision: {precision:.2f}")

recall = recall_score(y_test, y_pred, average='weighted', zero_division=1)
print(f"Recall: {recall:.2f}")

f1 = f1_score(y_test, y_pred, average='weighted', zero_division=1)
print(f"F1-Score: {f1:.2f}")

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=1))


Accuracy: 0.00
Precision: 1.00
Recall: 0.00
F1-Score: 0.00


MemoryError: Unable to allocate 2.11 GiB for an array with shape (16825, 16825) and data type int64

In [9]:

# K MEANS 

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

# Example training data (numeric features and ingredients list)
train_numeric_features = np.array([
    [15, 36, 1, 42, 21, 81, 2],  # Example numeric features
    [19, 18, 10, 73, 33, 104, 41],
    [22, 40, 3, 37, 25, 65, 9],
    [10, 12, 5, 50, 20, 72, 8]
])

train_ingredients = [
    'pork belly, smoked paprika, kosher salt',
    'sauerkraut, apples, pork loin',
    'garlic, chicken thighs, soy sauce',
    'onions, tomatoes, ground beef'
]

# Recipe names as labels
recipe_labels = ['Homemade Bacon', 'Pork Loin, Apples, and Sauerkraut', 'Soy Garlic Chicken', 'Beef Tomato Stew']

# Create a DataFrame to hold recipes (for precision calculation)
recipe_df = pd.DataFrame({
    'recipe_name': recipe_labels,
    'numeric_features': list(train_numeric_features),
    'ingredients': train_ingredients
})

# Fit the scaler on training numeric features
scaler = StandardScaler()
scaler.fit(train_numeric_features)

# Fit the vectorizer on training ingredients
vectorizer = TfidfVectorizer()
vectorizer.fit(train_ingredients)

# Transform the training data
train_numeric_features_scaled = scaler.transform(train_numeric_features)
train_ingredients_transformed = vectorizer.transform(train_ingredients).toarray()

# Combine the scaled numeric features and transformed ingredients for KNN training
train_combined_features = np.hstack([train_numeric_features_scaled, train_ingredients_transformed])

# Initialize and fit the KNN model
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(train_combined_features, recipe_labels)

# Define precision_at_k function
def precision_at_k(knn_model, test_features, test_ground_truth, k=2):
    precision_scores = []
    
    for input_features, relevant_recipes in zip(test_features, test_ground_truth):
        # Split the input into numeric features and ingredient features
        numeric_features = input_features[:7]  # First 7 are numeric
        ingredients = input_features[7]  # Last one is ingredient text
        
        # Scale numeric features and transform ingredients into vectorized form
        numeric_features_scaled = scaler.transform([numeric_features])
        ingredients_transformed = vectorizer.transform([ingredients])
        
        # Combine numeric and ingredient features into one input for KNN
        input_combined = np.hstack([numeric_features_scaled, ingredients_transformed.toarray()])
        
        # Get the top-k nearest neighbors from the model
        distances, indices = knn_model.kneighbors(input_combined, n_neighbors=k)
        recommended_recipes = recipe_df.iloc[indices[0]]['recipe_name'].values
        
        # Calculate precision: how many relevant recipes are among the recommended ones
        relevant_recommended = len(set(recommended_recipes).intersection(set(relevant_recipes)))
        precision = relevant_recommended / k
        
        precision_scores.append(precision)
    
    return np.mean(precision_scores)

# Example usage with test data
test_features = [
    [15, 36, 1, 42, 21, 81, 2, 'pork belly, smoked paprika, kosher salt'],  # Example input
    [19, 18, 10, 73, 33, 104, 41, 'sauerkraut, apples, pork loin']  # Another test case
]
test_ground_truth = [
    ['Homemade Bacon', 'Pork Loin, Apples, and Sauerkraut'],  # Expected relevant recipes for this test case
    ['Pork Loin, Apples, and Sauerkraut', 'Homemade Bacon']  # Another ground truth
]

# Call the precision function
precision_score = precision_at_k(knn, test_features, test_ground_truth, k=2)
print(f"Precision@2: {precision_score:.2f}")


Precision@2: 0.50


Testing The Algorithm

In [10]:
from sklearn.model_selection import train_test_split

# Assume 'ingredients_list' contains the textual ingredient data
# and the numerical columns you use are 'calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber'
recipe_df = pd.read_csv(file_path)

# Split into features (numerical + ingredients) and target (recipe names, etc.)
X = recipe_df[['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber', 'ingredients_list']]
y = recipe_df['recipe_name']

# Split the dataset into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Separate numerical features and ingredient lists
X_train_numerical = X_train[['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber']]
X_test_numerical = X_test[['calories', 'fat', 'carbohydrates', 'protein', 'cholesterol', 'sodium', 'fiber']]

# Normalize the numerical features
scaler = StandardScaler()
X_train_numerical_scaled = scaler.fit_transform(X_train_numerical)
X_test_numerical_scaled = scaler.transform(X_test_numerical)

# Vectorize the ingredients
vectorizer = TfidfVectorizer()
X_train_ingredients = vectorizer.fit_transform(X_train['ingredients_list'])
X_test_ingredients = vectorizer.transform(X_test['ingredients_list'])

# Combine numerical and ingredient features
X_train_combined = np.hstack([X_train_numerical_scaled, X_train_ingredients.toarray()])
X_test_combined = np.hstack([X_test_numerical_scaled, X_test_ingredients.toarray()])


In [12]:
from sklearn.neighbors import NearestNeighbors

# Train KNN model on the training set
knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(X_train_combined)


NearestNeighbors(metric='euclidean', n_neighbors=3)

In [13]:
# Recommend recipes for test data
def recommend_recipes(input_features):
    input_features_scaled = scaler.transform([input_features[:7]])
    input_ingredients_transformed = vectorizer.transform([input_features[7]])
    input_combined = np.hstack([input_features_scaled, input_ingredients_transformed.toarray()])
    distances, indices = knn.kneighbors(input_combined)
    recommendations = recipe_df.iloc[indices[0]]
    return recommendations[['recipe_name', 'ingredients_list', 'image_url']]

# Example usage on the test set (just one example from X_test)
test_input_features = X_test.iloc[0].tolist()  # This extracts features for the first test recipe
recommendations = recommend_recipes(test_input_features)
print(recommendations)


C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


                recipe_name  \
3272      Taco Turkey Wraps   
26426  Tart Lemon Triangles   
16399           Apple Toast   

                                        ingredients_list  \
3272   ['sour cream', 'pico de gallo salsa drained', ...   
26426  ['butter', "confectioners' sugar", 'all-purpos...   
16399  ['butter', 'white', 'cinnamon', 'large apple c...   

                                               image_url  
3272   https://images.media-allrecipes.com/userphotos...  
26426  http://images.media-allrecipes.com/userphotos/...  
16399  http://images.media-allrecipes.com/userphotos/...  


In [ ]:
# Generate ground truth based on ingredient similarity
print("hello")
y_test_relevant_recipes = []
for idx, test_ingredients in enumerate(X_test['ingredients_list']):
    # For each test recipe, find other recipes with overlapping ingredients in the dataset
    relevant = recipe_df[recipe_df['ingredients_list'].apply(lambda x: len(set(test_ingredients.split(', ')).intersection(set(x.split(', ')))) > 2)]
    
    # Store the recipe names as relevant recipes
    relevant_recipes = relevant['recipe_name'].tolist()
    y_test_relevant_recipes.append(relevant_recipes)
    
print(y_test_relevant_recipes)


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

# Function to calculate precision and recall at K
def precision_at_k(recommended, relevant, k):
    recommended_at_k = recommended[:k]
    relevant_at_k = len(set(recommended_at_k).intersection(relevant))
    return relevant_at_k / k

def recall_at_k(recommended, relevant, k):
    relevant_at_k = len(set(recommended[:k]).intersection(relevant))
    return relevant_at_k / len(relevant)

# Assume X_train_combined, X_test_combined, and y_test_relevant_recipes (ground truth) are already prepared
k = 3  # Number of top recommendations to consider

# Train KNN model on the training set
knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
knn.fit(X_train_combined)

# Evaluate model on the test set
precision_scores = []
recall_scores = []

for i in range(X_test_combined.shape[0]):
    input_combined = X_test_combined[i].reshape(1, -1)
    distances, indices = knn.kneighbors(input_combined, n_neighbors=k)
    
    # Get the recommended recipe names
    recommended_recipes = recipe_df.iloc[indices[0]]['recipe_name'].tolist()
    
    # Get the relevant recipes (ground truth) for this test input
    relevant_recipes = y_test_relevant_recipes[i]  # Replace with your ground truth list
    
    # Calculate Precision@K and Recall@K
    precision = precision_at_k(recommended_recipes, relevant_recipes, k)
    recall = recall_at_k(recommended_recipes, relevant_recipes, k)
    
    precision_scores.append(precision)
    recall_scores.append(recall)

# Average Precision@K and Recall@K across all test instances
avg_precision_at_k = np.mean(precision_scores)
avg_recall_at_k = np.mean(recall_scores)

print(f'Average Precision@{k}: {avg_precision_at_k:.2f}')
print(f'Average Recall@{k}: {avg_recall_at_k:.2f}')


Linear Regression

In [37]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression

# Example training data (numeric features and ingredients list)
train_numeric_features = np.array([
    [15, 36, 1, 42, 21, 81, 2],
    [19, 18, 10, 73, 33, 104, 41],
    [22, 40, 3, 37, 25, 65, 9],
    [10, 12, 5, 50, 20, 72, 8]
])

train_ingredients = [
    'pork belly, smoked paprika, kosher salt',
    'sauerkraut, apples, pork loin',
    'garlic, chicken thighs, soy sauce',
    'onions, tomatoes, ground beef'
]

# Recipe labels as numeric values (e.g., 0 = Homemade Bacon)
recipe_labels = [0, 1, 2, 3]  # Assigning a number to each recipe

# Create a DataFrame to hold recipes (for precision calculation)
recipe_df = pd.DataFrame({
    'recipe_name': ['Homemade Bacon', 'Pork Loin, Apples, and Sauerkraut', 'Soy Garlic Chicken', 'Beef Tomato Stew'],
    'numeric_label': recipe_labels
})

# Fit the scaler on training numeric features
scaler = StandardScaler()
scaler.fit(train_numeric_features)

# Fit the vectorizer on training ingredients
vectorizer = TfidfVectorizer()
vectorizer.fit(train_ingredients)

# Transform the training data
train_numeric_features_scaled = scaler.transform(train_numeric_features)
train_ingredients_transformed = vectorizer.transform(train_ingredients).toarray()

# Combine the scaled numeric features and transformed ingredients for model training
train_combined_features = np.hstack([train_numeric_features_scaled, train_ingredients_transformed])

# Initialize and fit the Linear Regression model
linear_reg = LinearRegression()
linear_reg.fit(train_combined_features, recipe_labels)

# Define precision_at_k function
def precision_at_k(model, test_features, test_ground_truth, k=2):
    precision_scores = []
    
    for input_features, relevant_recipes in zip(test_features, test_ground_truth):
        numeric_features = input_features[:7]
        ingredients = input_features[7]
        
        numeric_features_scaled = scaler.transform([numeric_features])
        ingredients_transformed = vectorizer.transform([ingredients])
        
        input_combined = np.hstack([numeric_features_scaled, ingredients_transformed.toarray()])
        
        # Predict the label with linear regression (round to nearest integer)
        predicted_label = int(round(model.predict(input_combined)[0]))
        
        # Find the corresponding recipe name from the predicted label
        predicted_recipe = recipe_df[recipe_df['numeric_label'] == predicted_label]['recipe_name'].values[0]
        
        # Calculate precision
        relevant_recommended = len(set([predicted_recipe]).intersection(set(relevant_recipes)))
        precision = relevant_recommended / k
        precision_scores.append(precision)
    
    return np.mean(precision_scores)

# Example usage with test data
test_features = [
    [15, 36, 1, 42, 21, 81, 2, 'pork belly, smoked paprika, kosher salt'],
    [19, 18, 10, 73, 33, 104, 41, 'sauerkraut, apples, pork loin']
]
test_ground_truth = [
    ['Homemade Bacon', 'Pork Loin, Apples, and Sauerkraut'],
    ['Pork Loin, Apples, and Sauerkraut', 'Homemade Bacon']
]

# Call the precision function
precision_score = precision_at_k(linear_reg, test_features, test_ground_truth, k=2)
print(f"Precision@2 (Linear Regression): {precision_score:.2f}")


Precision@2 (Linear Regression): 0.50


Decision Tree

In [38]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and fit the Decision Tree Classifier model
decision_tree = DecisionTreeClassifier()
decision_tree.fit(train_combined_features, recipe_labels)

# Define precision_at_k function
def precision_at_k(model, test_features, test_ground_truth, k=2):
    precision_scores = []
    
    for input_features, relevant_recipes in zip(test_features, test_ground_truth):
        numeric_features = input_features[:7]
        ingredients = input_features[7]
        
        numeric_features_scaled = scaler.transform([numeric_features])
        ingredients_transformed = vectorizer.transform([ingredients])
        
        input_combined = np.hstack([numeric_features_scaled, ingredients_transformed.toarray()])
        
        # Predict the label with decision tree
        predicted_label = model.predict(input_combined)[0]
        
        # Find the corresponding recipe name from the predicted label
        predicted_recipe = recipe_df[recipe_df['numeric_label'] == predicted_label]['recipe_name'].values[0]
        
        # Calculate precision
        relevant_recommended = len(set([predicted_recipe]).intersection(set(relevant_recipes)))
        precision = relevant_recommended / k
        precision_scores.append(precision)
    
    return np.mean(precision_scores)

# Call the precision function for Decision Tree
precision_score = precision_at_k(decision_tree, test_features, test_ground_truth, k=2)
print(f"Precision@2 (Decision Tree): {precision_score:.2f}")


Precision@2 (Decision Tree): 0.50
